# Financial Metrics - Latest WSB Stocks
This notebook refreshes financial metrics for the latest WallStreetBets tickers using current Yahoo Finance data.
It now:
- loads the latest top WSB stocks,
- fetches current Yahoo Finance price history,
- computes rolling standard deviation, SMA, EWMA, RSI, and MACD,
- runs a linear regression / RMSE check for each selected ticker,
- saves refreshed CSVs and charts under `outputs/latest_wsb_analysis/`.

## Step 1 - Setup

In [1]:
from pathlib import Path
import json
import sys
from typing import cast
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
try:
    ROOT = Path(__file__).resolve().parent
except NameError:
    ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.wsb_latest.pipeline import (
    compute_price_features,
    evaluate_linear_regression,
    fetch_price_history,
    plot_symbol_dashboard,
    run_pipeline,
)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
print(f"Project root: {ROOT}")

Project root: /home/runner/work/Project2-Sentiment-analysis-WSbets/Project2-Sentiment-analysis-WSbets


## Step 2 - Notebook configuration

In [2]:
OUTPUT_DIR = ROOT / "outputs" / "latest_wsb_analysis"
CHARTS_DIR = OUTPUT_DIR / "charts"
PRICES_DIR = OUTPUT_DIR / "prices"
PRICE_PERIOD = "1y"
TOP_K = 4
import os
REFRESH_WSB_DATA = os.getenv("WSB_REFRESH_DATA", "0") != "0"
MANUAL_TICKERS = None  # Example: ["AMD", "TSLA"]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHARTS_DIR.mkdir(parents=True, exist_ok=True)
PRICES_DIR.mkdir(parents=True, exist_ok=True)

## Step 3 - Refresh or load the latest WSB ranking

In [3]:
summary_path = OUTPUT_DIR / "top10_wsb_stocks.csv"
if REFRESH_WSB_DATA or (not summary_path.exists()):
    pipeline_result = run_pipeline(top_n=10, per_feed=100, price_period=PRICE_PERIOD, output_dir=OUTPUT_DIR)
    print(json.dumps(pipeline_result, indent=2))
summary_df = cast(pd.DataFrame, pd.read_csv(str(summary_path)))
summary_df["latest_mention"] = pd.to_datetime(summary_df["latest_mention"], utc=True)
summary_df[["ticker", "mention_count", "post_count", "avg_sentiment", "latest_mention"]]

,ticker,mention_count,post_count,avg_sentiment,latest_mention
0,RKLB,24,16,0.181250,2026-05-14 22:12:00+00:00
1,LNG,49,1,0.998300,2026-05-14 11:08:46+00:00
2,NBIS,12,10,0.485890,2026-05-14 17:55:59+00:00
3,NVDA,12,9,0.613533,2026-05-14 21:05:35+00:00
4,AMD,11,9,0.273544,2026-05-14 18:57:21+00:00
5,INTC,12,6,-0.105867,2026-05-11 22:50:15+00:00
6,SKM,31,1,0.997400,2026-05-13 18:56:48+00:00
7,NOW,7,4,0.347450,2026-05-14 18:37:57+00:00
8,PLTR,7,4,0.428225,2026-05-14 06:58:03+00:00
9,QQQ,5,4,0.335075,2026-05-14 14:16:39+00:00


## Step 4 - Select tickers for financial analysis

In [4]:
if MANUAL_TICKERS:
    selected_tickers = [ticker.upper() for ticker in MANUAL_TICKERS]
else:
    selected_tickers = summary_df["ticker"].head(TOP_K).tolist()
print("Selected tickers:", selected_tickers)

Selected tickers: ['RKLB', 'LNG', 'NBIS', 'NVDA']


## Step 5 - Fetch current price history and compute financial metrics

In [5]:
price_feature_frames = {}
regression_frames = {}
analysis_rows = []
for ticker in selected_tickers:
    history_df = fetch_price_history(ticker, PRICE_PERIOD)
    features_df = compute_price_features(history_df)
    rmse, regression_df = evaluate_linear_regression(features_df)
    features_path = PRICES_DIR / f"{ticker.lower()}_price_features.csv"
    regression_path = PRICES_DIR / f"{ticker.lower()}_regression_results.csv"
    dashboard_path = CHARTS_DIR / f"{ticker.lower()}_dashboard.png"
    features_df.to_csv(features_path)
    regression_df.to_csv(regression_path)
    plot_symbol_dashboard(ticker, features_df, regression_df, dashboard_path)
    price_feature_frames[ticker] = features_df
    regression_frames[ticker] = regression_df
    analysis_rows.append({
        "ticker": ticker,
        "rows": int(len(features_df)),
        "last_close": float(features_df["Close"].dropna().iloc[-1]),
        "latest_rsi": float(features_df["RSI"].dropna().iloc[-1]),
        "latest_macd": float(features_df["MACD"].dropna().iloc[-1]),
        "return_21d": float(features_df["21D Return"].dropna().iloc[-1]),
        "rmse": rmse,
        "features_csv": str(features_path),
        "regression_csv": str(regression_path),
        "dashboard_png": str(dashboard_path),
    })
analysis_df = pd.DataFrame(analysis_rows).sort_values("ticker").reset_index(drop=True)
analysis_summary_path = OUTPUT_DIR / "financial_metrics_summary.csv"
analysis_df.to_csv(analysis_summary_path, index=False)
analysis_df

,ticker,rows,last_close,latest_rsi,latest_macd,return_21d,rmse,features_csv,regression_csv,dashboard_png
0,LNG,251,241.080002,36.926480,-5.860183,-0.061032,3.977198,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...
1,NBIS,251,221.149994,72.838635,18.339010,0.326078,6.930036,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...
2,NVDA,251,235.740005,76.719554,9.244225,0.185398,2.629439,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...
3,RKLB,251,132.550003,78.332955,12.034796,0.800951,4.634048,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...


## Step 6 - Inspect the computed metrics for each selected ticker

In [6]:
for ticker in selected_tickers:
    print(f"\n===== {ticker} latest metrics =====")
    display_cols = [
        "Close",
        "Volume",
        "30-Day Rolling STD",
        "30-Day Rolling SMA",
        "30-Day Rolling EWMA",
        "RSI",
        "MACD",
        "5D Return",
        "21D Return",
    ]
    print(price_feature_frames[ticker][display_cols].tail(5).to_string())


===== RKLB latest metrics =====
                 Close    Volume  30-Day Rolling STD  30-Day Rolling SMA  30-Day Rolling EWMA        RSI       MACD  5D Return  21D Return
Date                                                                                                                                      
2026-05-08  105.470001  79927300           10.289285           76.507334            73.550633  68.750255   4.085717   0.338282    0.580312
2026-05-11  117.349998  54779700           12.303035           78.388000            74.554270  73.376079   6.433867   0.461213    0.724467
2026-05-12  117.559998  34195200           13.597621           80.394000            75.539649  73.450893   8.217018   0.492636    0.664684
2026-05-13  124.150002  26587400           15.419684           82.391667            76.653364  75.753444  10.046130   0.466627    0.719053
2026-05-14  132.550003  24888843           17.593759           84.626000            77.933924  78.332955  12.034796   0.686816    0.8

## Step 7 - Plot the latest closing prices for the selected tickers

In [7]:
fig, ax = plt.subplots(figsize=(12, 6))
for ticker in selected_tickers:
    close_series = price_feature_frames[ticker]["Close"].dropna()
    ax.plot(close_series.index, close_series.values, label=ticker, linewidth=1.8)
ax.set_title("Latest selected WSB tickers - close prices")
ax.set_ylabel("Close")
ax.legend(loc="upper left")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Step 8 - Review the regression predictions

In [8]:
for ticker in selected_tickers:
    regression_df = regression_frames[ticker]
    print(f"\n===== {ticker} regression results =====")
    if regression_df.empty:
        print("Not enough data for regression output.")
    else:
        print(regression_df.tail(10).to_string())


===== RKLB regression results =====
                 Close  Predicted Close
Date                                   
2026-05-01   78.809998        84.453092
2026-05-04   80.309998        84.460410
2026-05-05   78.760002        84.141559
2026-05-06   84.650002        87.373564
2026-05-07   78.580002        83.673049
2026-05-08  105.470001        97.902424
2026-05-11  117.349998       106.141330
2026-05-12  117.559998       111.402445
2026-05-13  124.150002       116.572003
2026-05-14  132.550003       125.947643

===== LNG regression results =====
                 Close  Predicted Close
Date                                   
2026-05-01  270.059998       271.408785
2026-05-04  272.850006       272.838050
2026-05-05  269.519989       269.318701
2026-05-06  261.420013       262.743310
2026-05-07  246.779999       254.602743
2026-05-08  240.110001       248.666323
2026-05-11  240.699997       244.829042
2026-05-12  244.309998       245.199570
2026-05-13  239.380005       242.508591
2026-05

## Step 9 - Final notebook summary

In [9]:
notebook_summary = {
    "selected_tickers": selected_tickers,
    "price_period": PRICE_PERIOD,
    "top_k": TOP_K,
    "analysis_summary_csv": str(analysis_summary_path.resolve()),
    "output_dir": str(OUTPUT_DIR.resolve()),
}
print(json.dumps(notebook_summary, indent=2))

{
  "selected_tickers": [
    "RKLB",
    "LNG",
    "NBIS",
    "NVDA"
  ],
  "price_period": "1y",
  "top_k": 4,
  "analysis_summary_csv": "/home/runner/work/Project2-Sentiment-analysis-WSbets/Project2-Sentiment-analysis-WSbets/outputs/latest_wsb_analysis/financial_metrics_summary.csv",
  "output_dir": "/home/runner/work/Project2-Sentiment-analysis-WSbets/Project2-Sentiment-analysis-WSbets/outputs/latest_wsb_analysis"
}
